# SDC Manifest Builder

Reads **only the first and last records** of each SDC file to extract `t_min` / `t_max`  
without loading the full point cloud into RAM.

SDC timestamps are in **SOW (Seconds of Week)**. This notebook converts them to  
**GPS seconds** using: `t_gps = t_sow + day_shift * 86400 + leapsec`  
so that manifest values are directly comparable to `georef_time_window` bounds.

Outputs one CSV per scanner:
- `manifest_HA.csv`
- `manifest_LR.csv`
- `manifest_PUCK.csv`

These manifests are then referenced in the scanner YAML configs so that  
`pointCloudGeoref.py` only processes SDC files that overlap the time window of interest.

In [1]:
import struct
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

## Configuration

In [7]:
SCANNER_DIRS = {
    "HA":   Path("/media/b085164/Elements/CALIB_26_02_25/VUX_SDC/HA/"),
    "LR":   Path("/media/b085164/Elements/CALIB_26_02_25/VUX_SDC/LR/"),
    "PUCK": Path("/media/b085164/Elements/CALIB_26_02_25/PUCK_SDC/"),
}

TIME_CORRECTIONS = {
    "HA":   {"day_shift": 3, "leapsec": 18},
    "LR":   {"day_shift": 3, "leapsec": 18},
    # "PUCK": {"day_shift": 0, "leapsec": 18},
}

MANIFEST_DIR = Path("/media/b085164/Elements/CALIB_26_02_25/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

N_EDGE = 500

## SDC record layout

In [8]:
RECORD_DTYPE = np.dtype([
    ("t",    "<f8"),
    ("f1",   "<f4"),
    ("f2",   "<f4"),
    ("x",    "<f4"),
    ("y",    "<f4"),
    ("z",    "<f4"),
    ("u6",   "<u2"),
    ("u7",   "<u2"),
    ("u8",   "u1"),
    ("u9",   "u1"),
    ("u10",  "u1"),
    ("u11",  "<u2"),
    ("u12",  "u1"),
    ("u13",  "u1"),
    ("f14",  "<f4"),
    ("u15",  "<u2"),
])
RECORD_SIZE = RECORD_DTYPE.itemsize
print(f"Record size: {RECORD_SIZE} bytes")


def sow_to_gps(t_sow: np.ndarray, day_shift: int, leapsec: float) -> np.ndarray:
    return t_sow + day_shift * 86400.0 + leapsec


def sdc_tmin_tmax(sdc_path: Path, day_shift: int = 0, leapsec: float = 0.0, n_edge: int = N_EDGE) -> dict:
    result = {
        "filename":  sdc_path.name,
        "t_min_sow": np.nan,
        "t_max_sow": np.nan,
        "t_min":     np.nan,
        "t_max":     np.nan,
        "n_records": 0,
        "error":     None,
    }
    try:
        file_size = sdc_path.stat().st_size
        with open(sdc_path, "rb") as f:
            header_bytes = f.read(4)
        size_of_header = struct.unpack("<I", header_bytes)[0]

        payload_size = file_size - size_of_header
        if payload_size <= 0 or payload_size % RECORD_SIZE != 0:
            result["error"] = (
                f"Invalid payload: {payload_size} bytes "
                f"(header={size_of_header}, record={RECORD_SIZE})"
            )
            return result

        n_records = payload_size // RECORD_SIZE
        result["n_records"] = int(n_records)
        n_read = min(n_edge, n_records)

        with open(sdc_path, "rb") as f:
            f.seek(size_of_header)
            head = np.frombuffer(f.read(n_read * RECORD_SIZE), dtype=RECORD_DTYPE)
            t_head = head["t"]
            if n_records > n_read:
                tail_offset = size_of_header + (n_records - n_read) * RECORD_SIZE
                f.seek(tail_offset)
                tail = np.frombuffer(f.read(n_read * RECORD_SIZE), dtype=RECORD_DTYPE)
                t_tail = tail["t"]
            else:
                t_tail = t_head

        t_all = np.concatenate([t_head, t_tail])
        mask = np.isfinite(t_all) & (t_all > 0) & (t_all < 1e8)
        if not np.any(mask):
            result["error"] = "No finite timestamps found in edge records"
            return result

        t_valid = t_all[mask]
        t_min_sow = float(np.min(t_valid))
        t_max_sow = float(np.max(t_valid))
        result["t_min_sow"] = t_min_sow
        result["t_max_sow"] = t_max_sow
        result["t_min"]     = float(sow_to_gps(np.array([t_min_sow]), day_shift, leapsec)[0])
        result["t_max"]     = float(sow_to_gps(np.array([t_max_sow]), day_shift, leapsec)[0])

    except Exception as e:
        result["error"] = str(e)

    return result

def csv_tmin_tmax(csv_path: Path, day_shift: int = 0, leapsec: float = 0.0, skiprows: int = 1) -> dict:
    result = {
        "filename":  csv_path.name,
        "t_min_sow": np.nan,
        "t_max_sow": np.nan,
        "t_min":     np.nan,
        "t_max":     np.nan,
        "n_records": 0,
        "error":     None,
    }
    try:
        with open(csv_path, "rb") as f:
            n_total = sum(1 for _ in f) - skiprows
        result["n_records"] = int(n_total)
        n_read = min(N_EDGE, n_total)

        head = pd.read_csv(
            csv_path, usecols=[3], header=None,
            skiprows=skiprows, nrows=n_read, dtype=np.float64,
        ).iloc[:, 0].values

        if n_total > n_read:
            skip_until = skiprows + n_total - n_read
            tail = pd.read_csv(
                csv_path, usecols=[3], header=None,
                skiprows=lambda i: i < skip_until,
                dtype=np.float64,
            ).iloc[:, 0].values
        else:
            tail = head

        t_all = np.concatenate([head, tail])
        mask = np.isfinite(t_all) & (t_all > 0) & (t_all < 1e8)
        if not np.any(mask):
            result["error"] = "No finite timestamps found"
            return result

        t_valid = t_all[mask]
        t_min_sow = float(np.min(t_valid))
        t_max_sow = float(np.max(t_valid))
        result["t_min_sow"] = t_min_sow
        result["t_max_sow"] = t_max_sow
        result["t_min"]     = t_min_sow + leapsec   # pas de day_shift pour CSV
        result["t_max"]     = t_max_sow + leapsec

    except Exception as e:
        result["error"] = str(e)
    return result


Record size: 45 bytes


## Build manifests for all scanners

In [10]:
manifest_paths = {}

for scanner_name, scan_dir in SCANNER_DIRS.items():
    print(f"\n{'='*60}")
    print(f"Scanner: {scanner_name}  —  {scan_dir}")
    print(f"{'='*60}")

    if not scan_dir.exists():
        print(f"  [SKIP] Directory not found: {scan_dir}")
        continue

    sdc_files = sorted(scan_dir.glob("*.sdc"))
    csv_files = sorted(scan_dir.glob("*.csv"))
    all_files = sdc_files + csv_files

    if not all_files:
        print(f"  [SKIP] No .sdc or .csv files found in {scan_dir}")
        continue

    print(f"  Found {len(sdc_files)} SDC + {len(csv_files)} CSV files")

    tc        = TIME_CORRECTIONS[scanner_name]
    day_shift = tc["day_shift"]
    leapsec   = tc["leapsec"]

    rows   = []
    errors = []

    for fpath in tqdm(all_files, desc=scanner_name, unit="file"):
        if fpath.suffix.lower() == ".sdc":
            info = sdc_tmin_tmax(fpath, day_shift=day_shift, leapsec=leapsec)
        else:
            info = csv_tmin_tmax(fpath)   # GPS SOW direct, pas de correction
        rows.append(info)
        if info["error"]:
            errors.append((fpath.name, info["error"]))

    df = pd.DataFrame(rows, columns=["filename", "t_min_sow", "t_max_sow", "t_min", "t_max", "n_records", "error"])
    df = df.sort_values("t_min").reset_index(drop=True)

    out_path = MANIFEST_DIR / f"manifest_{scanner_name}.csv"
    df[["filename", "t_min_sow", "t_max_sow", "t_min", "t_max", "n_records"]].to_csv(
        out_path, index=False, float_format="%.6f"
    )
    manifest_paths[scanner_name] = out_path

    print(f"  Saved: {out_path}")
    print(f"  GPS range : [{df['t_min'].min():.3f}, {df['t_max'].max():.3f}]")

    if errors:
        print(f"  ERRORS ({len(errors)}):")
        for fname, err in errors:
            print(f"    {fname}: {err}")

print("\nDone.")


Scanner: HA  —  /media/b085164/Elements/CALIB_26_02_25/VUX_SDC/HA
  Found 28 SDC + 0 CSV files


HA:   0%|          | 0/28 [00:00<?, ?file/s]

  Saved: /media/b085164/Elements/CALIB_26_02_25/manifests/manifest_HA.csv
  GPS range : [305005.361, 306650.773]

Scanner: LR  —  /media/b085164/Elements/CALIB_26_02_25/VUX_SDC/LR
  Found 28 SDC + 0 CSV files


LR:   0%|          | 0/28 [00:00<?, ?file/s]

  Saved: /media/b085164/Elements/CALIB_26_02_25/manifests/manifest_LR.csv
  GPS range : [305005.129, 306650.774]

Scanner: PUCK  —  /media/b085164/Elements/CALIB_26_02_25/PUCK_SDC
  Found 2 SDC + 0 CSV files


KeyError: 'PUCK'

## Preview manifests

In [6]:
for scanner_name, csv_path in manifest_paths.items():
    df = pd.read_csv(csv_path)
    print(f"\n--- {scanner_name} ({len(df)} files) ---")
    display(df.head(10))


--- HA (20 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_152829_VUX-HA1.sdc,55710.111174,55792.241952,314928.111174,315010.241952,53313151
1,260225_153059_VUX-HA1.sdc,55860.291329,55878.943128,315078.291329,315096.943128,14783955
2,260225_153206_VUX-HA1.sdc,55927.126311,55942.146213,315145.126311,315160.146213,9838149
3,260225_153229_VUX-HA1.sdc,55950.025743,56002.284617,315168.025743,315220.284617,35805276
4,260225_153524_VUX-HA1.sdc,56124.955888,56167.613151,315342.955888,315385.613151,29352701
5,260225_153639_VUX-HA1.sdc,56200.185105,56232.647148,315418.185105,315450.647148,22131660
6,260225_153842_VUX-HA1.sdc,56323.086140,56347.418145,315541.086140,315565.418145,16045061
7,260225_154000_VUX-HA1.sdc,56401.101034,56441.282353,315619.101034,315659.282353,32669974
8,260225_154111_VUX-HA1.sdc,56471.861078,56499.151474,315689.861078,315717.151474,17256935
9,260225_154155_VUX-HA1.sdc,56515.765383,56549.381664,315733.765383,315767.381664,22420357



--- LR (20 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_152829_VUX1-LR.sdc,55710.118766,55792.240510,314928.118766,315010.240510,44766186
1,260225_153059_VUX1-LR.sdc,55860.303337,55878.946288,315078.303337,315096.946288,12722511
2,260225_153206_VUX1-LR.sdc,55927.073348,55942.146456,315145.073348,315160.146456,8842159
3,260225_153229_VUX1-LR.sdc,55950.268421,56002.286969,315168.268421,315220.286969,29794241
4,260225_153524_VUX1-LR.sdc,56124.847700,56167.615266,315342.847700,315385.615266,24493576
5,260225_153639_VUX1-LR.sdc,56200.113087,56232.640248,315418.113087,315450.640248,19001965
6,260225_153842_VUX1-LR.sdc,56323.328752,56347.419054,315541.328752,315565.419054,13764474
7,260225_154000_VUX1-LR.sdc,56401.152636,56441.276498,315619.152636,315659.276498,27190359
8,260225_154111_VUX1-LR.sdc,56471.858615,56499.152924,315689.858615,315717.152924,14068400
9,260225_154155_VUX1-LR.sdc,56515.783667,56549.380920,315733.783667,315767.380920,17477970



--- PUCK (5 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,lidar_20260225_153905.sdc,0.000000,9.145079e+07,18.000000,9.145080e+07,95751038
1,lidar_20260225_160632.sdc,0.000000,3.171747e+05,18.000000,3.171927e+05,122985831
2,lidar_20260225_152758.sdc,314860.677110,3.150889e+05,314878.677110,3.151069e+05,28015814
3,lidar_20260225_153232.sdc,315134.126494,3.152149e+05,315152.126494,3.152329e+05,12639794
4,lidar_20260225_153532.sdc,315314.197358,3.154449e+05,315332.197358,3.154629e+05,22485094


## (Optional) Filter preview — which files fall in a time window?

Useful to verify before running the full georef pipeline.

In [7]:
# --- Preview: which SDCs overlap the outage window? ---
outage_t_start = 305120.0
outage_duration = 475.0
margin_s = 60.0

t_lo = outage_t_start - margin_s
t_hi = outage_t_start + outage_duration + margin_s

print(f"Window: [{t_lo:.1f}, {t_hi:.1f}]  ({(t_hi - t_lo):.0f} s)")
print()

for scanner_name, csv_path in manifest_paths.items():
    df = pd.read_csv(csv_path)
    # a file overlaps the window if its t_max >= t_lo AND t_min <= t_hi
    mask = (df["t_max"] >= t_lo) & (df["t_min"] <= t_hi)
    print(f"  {scanner_name}: {mask.sum()}/{len(df)} files overlap the window")
    display(df[mask])

Window: [305060.0, 305655.0]  (595 s)

  HA: 0/20 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records


  LR: 0/20 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records


  PUCK: 2/5 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,lidar_20260225_153905.sdc,0.0,9.145079e+07,18.0,9.145080e+07,95751038
1,lidar_20260225_160632.sdc,0.0,3.171747e+05,18.0,3.171927e+05,122985831
